# Setup

In [ ]:
!git clone https://github.com/11erlangga/erlangga_kezia_naurah.git
!pip install catboost optuna
%cd erlangga_kezia_naurah

In [ ]:
GPU_PARAMS = {
    "task_type": "GPU",
    "devices": "0",
}

# Import Dependencies

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from optuna.visualization.matplotlib import plot_optimization_history
from sklearn.model_selection import train_test_split

from src.features import add_missing_indicators
from src.modeling.tuning import tune_catboost

# Load Data & Reproduksi Feature Set

In [ ]:
# Load dataset
df = pd.read_csv("data/raw/train.csv", index_col="sample_id")
df.head()

In [ ]:
# Pembuatan fitur indikator
df = add_missing_indicators(df)

In [ ]:
# Split dataset
X = df.drop(columns="property_organic_content")
y = df["property_organic_content"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=X["source_id"], random_state=42)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
# Categorical fetures
cat_feats = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

# Jalankan Optuna Study

In [ ]:
study = tune_catboost(X=X_train, y=y_train, cat_feats=cat_feats, n_trials=100, n_splits=4, extra_params=GPU_PARAMS)

# Diagnostik Hasil Tuning

In [ ]:
print("Best RMSE :", round(study.best_value, 6))
print("Best params:")
for k, v in study.best_params.items():
    print(f"--> {k}: {v}")

plot_optimization_history(study)
plt.tight_layout()
plt.show()

# Simpan Hasil Study

In [ ]:
joblib.dump(study, "outputs/study.pkl")